# HLD/ETH — Analyse de micro-structure (Uniswap V4, Base)

**Données on-chain réelles** extraites via `extract_live.py`.

⚠️  Les courbes de coût d'exécution (slippage) sont des **sorties de modèle** constant-product
(x*y=k) appliquées aux réserves réelles. Ce ne sont **PAS des transactions observées**.
Toujours lire « coût d'exécution modélisé ».

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent
with open(ROOT / "data" / "pool_state.json") as f:
    state = json.load(f)

Q96 = 2**96
L = state["liquidity"]
sqrtPX96 = state["sqrtPriceX96"]
tick = state["tick"]
eth_eur = state.get("eth_eur", 1645.88)

sqrtP = sqrtPX96 / Q96
spot_hld_per_eth = sqrtP ** 2
spot_eth_per_hld = 1 / spot_hld_per_eth

print(f"Bloc: #{state['block']} ({state['timestamp_utc'][:10]})")
print(f"Tick: {tick}")
print(f"lpFee: {state['lpFee']} pips ({state['lpFee']/10000:.1f}%)")
print(f"L: {L:,}")
print(f"1 ETH = {spot_hld_per_eth:,.0f} HLD")
print(f"1 HLD = {spot_eth_per_hld:.4e} ETH")
print(f"1 M HLD ≈ {spot_eth_per_hld * eth_eur * 1e6:.4f} €")
print(f"ETH/EUR = {eth_eur:.2f} €")

In [ ]:
# Réserves virtuelles (modèle constant-product)
x_virt = L / sqrtP  # ETH
y_virt = L * sqrtP  # HLD
k = x_virt * y_virt
pool_value_eth = 2 * x_virt

print(f"x_virt (ETH)  = {x_virt:,.6f}")
print(f"y_virt (HLD)  = {y_virt:,.0f}")
print(f"k             = x*y = {k:,.0f}")
print(f"Valeur pool   ≈ {pool_value_eth * eth_eur:,.0f} €")

In [ ]:
# Coût d'exécution MODÉLISÉ
def cp_slippage(x, y, dy):
    """Coût pour acheter dy HLD (envoyer ETH, recevoir HLD)."""
    new_y = y - dy
    new_x = (x * y) / new_y
    dx = new_x - x
    avg_price = dy / dx  # HLD/ETH
    spot = y / x
    cost_bps = (avg_price / spot - 1) * 10000
    return avg_price, cost_bps, dx

# Courbe de coût
sizes = np.logspace(1, np.log10(y_virt * 0.9), 300)
costs = []
for s in sizes:
    _, c, _ = cp_slippage(x_virt, y_virt, s)
    costs.append(min(c, 10000))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Courbe complète
ax1.plot(sizes / 1e6, costs, linewidth=2, color='#2172E5')
ax1.set_xscale('log'); ax1.set_yscale('log')
ax1.set_xlabel('Trade (millions de HLD)')
ax1.set_ylabel('Coût d\'exécution modélisé (bps)')
ax1.set_title('Coût d\'exécution modélisé (échelle log-log)')
ax1.axhline(100, color='orange', ls='--', alpha=0.5, label='lpFee (1%)')
ax1.grid(True, alpha=0.3, which='both')
ax1.legend()

# Zoom <1000 bps
mask = np.array(costs) < 1000
ax2.plot(sizes[mask] / 1e6, np.array(costs)[mask], linewidth=2, color='#2172E5')
ax2.set_xlabel('Trade (millions de HLD)')
ax2.set_ylabel('Coût d\'exécution modélisé (bps)')
ax2.set_title('Coût d\'exécution modélisé (zoom)')
ax2.axhline(100, color='orange', ls='--', alpha=0.5, label='lpFee (1%)')
ax2.grid(True, alpha=0.3)
ax2.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Vérification: cohérence tick ⇔ sqrtPriceX96
price_from_tick = 1.0001 ** tick
ratio = price_from_tick / spot_hld_per_eth
assert abs(ratio - 1) < 0.01, f"Incohérence tick/prix: ratio={ratio:.6f}"
print(f"✓ Cohérence tick ⇔ sqrtPriceX96: ratio = {ratio:.6f}")

## Conclusion

- Les données de prix et liquidité sont **réelles** (on-chain, Base bloc #49023454).
- Les courbes de coût d'exécution sont des **sorties de modèle** constant-product.
- **1 HLD ≈ 0.000075 €** — token à micro-capitalisation.
- Pool avec **1% de frais**, liquidité modeste (~4.45e20).